Отличный вопрос! Вы перечислили ключевые современные алгоритмы для решения задач оптимальной остановки, особенно в контексте оценки американских опционов в финансовой математике. Давайте расшифруем и подробно разберем каждый из них.

### Общая постановка задачи
Все эти методы решают одну проблему: **нахождение оптимального момента для выполнения действия** (например, исполнения опциона), чтобы максимизировать ожидаемую выгоду в будущем. Основная сложность — это "проклятие размерности", когда число состояний системы (например, траекторий моделируемых цены активов) огромно.

---

### 1. LSM (Least Squares Monte Carlo)
*   **Авторы и год:** Longstaff and Schwartz (2001)
*   **Суть:** Классический и самый известный метод. Это **основа, с которой сравнивают все остальные**.
*   **Как работает:**
    1.  Генерируется множество (например, 100 000) возможных будущих траекторий цены базового актива (Монте-Карло).
    2.  **Идет "назад во времени"**, начиная с даты экспирации.
    3.  На каждом шаге назад для каждого сценария ("пути") вычисляется **непосредственная выплата** от немедленного исполнения.
    4.  **Ключевой шаг:** Для всех путей, где опцион "в деньгах" (есть смысл рассматривать исполнение), строится регрессия (методом наименьших квадратов) между *непосредственной выплатой* и текущими *состояниями рынка* (например, цена актива, волатильность). Это позволяет **оценить условное математическое ожидание будущей выгоды** от продолжения удержания опциона.
    5.  **Правило остановки:** Если непосредственная выплата больше, чем оцененная стоимость продолжения, опцион исполняется на этом пути в данный момент. В противном случае — продолжается.
    6.  Процесс повторяется до начальной даты.
*   **Плюсы:** Интуитивно понятный, относительно прост в реализации.
*   **Минусы:** Точность критически зависит от выбора **базисных функций** для регрессии (например, полиномы Лагерра, степенные ряды). При высокой размерности (много факторов) подобрать хорошие функции сложно.

---

### 2. NLSM (Neural Least Squares Monte Carlo) и DOS (Deep Optimal Stopping)
*   **Авторы и годы:**
    *   NLSM: Lapeyre & Lelong (2019) и Becker, Cheridito & Jentzen (2019).
    *   DOS: Тот же трио — Becker, Cheridito & Jentzen (2019) — часто это название для их конкретной реализации NLSM.
*   **Суть:** **Прямое развитие LSM, где вместо простой линейной регрессии на предзаданных базисных функциях используется нейронная сеть.**
*   **Как работает (общая идея):**
    1.  Так же генерируются траектории Монте-Карло.
    2.  Так же процесс идет назад во времени.
    3.  **Вместо регрессии:** На каждом временном шаге (или используя одну общую рекуррентную сеть) обучается небольшая нейронная сеть. **На вход** ей подаются текущие состояния (цены и т.д.), а **на выходе** — предсказание условного математического ожидания стоимости продолжения.
    4.  Нейронная сеть — это **универсальный аппроксиматор**, который сам учится из данных находить наилучшие "признаки" (заменяющие базисные функции в LSM) для оценки.
    5.  Правило остановки аналогично: сравнение непосредственной выплаты и прогноза сети.
*   **Плюсы:**
    *   **Автоматическое извлечение признаков.** Не нужно вручную подбирать базисные функции.
    *   **Лучшая масштабируемость в высоких размерностях.** Эффективно работает, когда факторов риска много (например, в опционах на корзину активов).
*   **Минусы:** Требует больше вычислительных ресурсов для обучения, есть вопросы выбора архитектуры сети и риска переобучения.

---

### 3. RLSM (Randomized Least Squares Monte Carlo)
*   **Авторы и год:** Herrera, Krach, Ruyssen & Teichmann (2021)
*   **Суть:** **"Смешанный" подход**, который пытается объединить скорость LSM и мощь нейросетей, **избегая пошагового обучения во времени**.
*   **Как работает (ключевое отличие):**
    1.  Генерируются траектории.
    2.  Вместо обучения разных моделей на каждом шаге времени **обучается одна глобальная параметрическая функция** (например, нейронная сеть, но проще, чем в NLSM). Ее задача — предсказать **всю будущую оптимальную стратегию остановки**.
    3.  **Сердце метода — рандомизация:** На каждом пути и на каждом временном шаге генерируется **случайная метка продолжения/остановки**. Затем методом наименьших квадратов (регрессией) подбираются параметры сети так, чтобы ее предсказания наиболее точно **соответствовали этим меткам, взвешенным по фактической выгоде**.
    4.  Фактически, алгоритм в один заход находит функцию, которая для любого состояния в любой момент времени выдает вероятность того, что остановка в этом состоянии оптимальна.
*   **Плюсы:**
    *   **Вычислительная эффективность.** Одно обучение вместо множества (как в LSM/NLSM).
    *   **Глобальная оптимизация.** Стратегия ищется сразу для всего пространства состояний-времени, а не конструируется пошагово.
    *   Меньше риск переобучения на отдельных временных шагах.
*   **Минусы:** Менее интуитивен, чем LSM. Требует аккуратной настройки процесса рандомизации меток.

---

### Сводная таблица

| Метод | Ключевая идея | "Двигатель" оценки продолжения | Обучение | Основное преимущество |
| :--- | :--- | :--- | :--- | :--- |
| **LSM** | Обратная индукция + регрессия | Линейная регрессия на **предзаданных** базисных функциях | Пошаговое, на каждом временном слое | Простота, эталонный метод |
| **NLSM/DOS**| Обратная индукция + глубокое обучение | **Нейронная сеть** (универсальный аппроксиматор) | Пошаговое, на каждом временном слое | Автоматическое извлечение признаков, мощность в высоких размерностях |
| **RLSM** | Глобальная оптимизация + рандомизация | Параметрическая функция (например, небольшая сеть), обучаемая **по методу наименьших квадратов** | **Единое глобальное** обучение | Вычислительная скорость, единая стратегия для всех времен |

**Вывод:** Эволюция методов идет от ручного подбора признаков (LSM) к их автоматическому обучению (NLSM) и далее к оптимизации самой процедуры обучения для ускорения вычислений (RLSM). Все они являются мощными инструментами для решения сложных задач оптимальной остановки в финансах и не только.

Вот исправленная и готовая для презентации таблица с очень краткими формулировками, а затем — скрипт для устного рассказа.

### Готовая таблица для слайда

| Метод | Ключевая идея | "Двигатель" оценки | Обучение | Ключевое преимущество |
| :--- | :--- | :--- | :--- | :--- |
| **LSM** | Обратная индукция + регрессия | Предзаданные базисные функции | Пошаговое  | **Простота и надёжность.** Эталон для сравнений. |
| **NLSM/DOS**| Обратная индукция + ИИ | Нейронная сеть | Пошаговое | **Мощность.** Автоматически находит сложные зависимости в данных. |
| **RLSM** | Глобальная оптимизация + рандомизация | Одна параметрическая модель | **Единое** (глобальное) | **Скорость.** Быстро находит стратегию для всех сценариев. |

---

### Текст для рассказа (пояснение слайда)

**Вступление (перед показом таблицы):**
> "Для решения задач оптимальной остановки, особенно в высоких размерностях, классический метод LSM стал отправной точкой. За последние годы появились его мощные модернизации, которые мы сейчас сравним."

**Пояснение строк таблицы (зачитывайте, последовательно показывая на каждую строку):**

1.  **Про LSM:**
    > "**LSM — это классика.** Алгоритм идёт назад во времени и на каждом шаге с помощью простой линейной регрессии оценивает, стоит ли продолжать. Его сила — в простоте и понятности, поэтому он до сих пор является эталоном. Но есть и слабость: для регрессии нужно вручную подбирать 'ингредиенты' — базисные функции, что в сложных задачах становится искусством."

2.  **Про NLSM/DOS:**
    > "**NLSM или Deep Optimal Stopping — это эволюция LSM с использованием глубокого обучения.** Вместо ручного подбора функций здесь на каждом шаге работает нейронная сеть. Она сама учится извлекать из данных сложные закономерности. Это делает метод исключительно мощным для задач с множеством факторов, где LSM уже не справляется. По сути, мы меняем ручную настройку на автоматическое обучение."

3.  **Про RLSM:**
    > "**RLSM — это следующий шаг, который меняет сам принцип обучения.** Если первые два метода учат стратегию шаг за шагом, то RLSM обучает одну общую модель сразу для всех моментов времени. Он делает это с помощью clever-приёма — рандомизации меток 'продолжить/остановиться' — и последующей глобальной оптимизации. Главный выигрыш здесь — в скорости: стратегия находится за один проход, а не конструируется по кусочкам."

**Заключение (после таблицы):**
> "Таким образом, мы видим эволюцию: от **простого и контролируемого LSM**, через **мощный и гибкий NLSM**, к **быстрому и глобальному RLSM**. Выбор метода зависит от задачи: нужен ли нам проверенный эталон, максимальная точность в сложных условиях или высокая скорость вычислений."

Этот текст и таблица создают четкую логическую цепочку и хорошо подходят для формата презентации.

Ниже — **чёткое мотивационное объяснение, зачем вообще нужен RLSM**, а затем — **наглядная схема его работы**, которую можно либо **перерисовать на слайд**, либо использовать как основу для картинки.

---

## Зачем вообще нужен RLSM

На первый взгляд RLSM выглядит как «хуже, чем LSM и NLSM», но это **неверная интерпретация**.
RLSM появился как **ответ на реальные вычислительные ограничения**, а не как метод для максимальной точности.

### Ключевая мотивация RLSM

**LSM и NLSM плохо масштабируются**, когда:

* число временных шагов большое;
* размерность состояния высокая;
* нужно много раз переоценивать опцион при разных параметрах;
* важна скорость, а не максимальная точность.

RLSM решает именно эти проблемы.

---

### Что именно улучшает RLSM

1. **Убирает пошаговое обучение по времени**
   В LSM / NLSM:

   * на каждом шаге времени обучается новая регрессия / сеть;
   * стоимость растёт линейно с числом шагов.

   В RLSM:

   * используется **одна глобальная модель**;
   * обучение проводится **один раз**.

2. **Фиксирует представление признаков**

   * случайный базис заменяет ручной выбор функций;
   * нет градиентного обучения скрытого слоя;
   * резко падает вычислительная стоимость.

3. **Хорошо работает в “индустриальном” режиме**

   * много сценариев;
   * быстрое приближённое решение;
   * предварительная фильтрация стратегий.

---

## Интуитивно: где место RLSM

> **RLSM — это метод “быстро и достаточно хорошо”,
> а не “максимально точно”.**

Он нужен:

* как baseline в высокоразмерных задачах;
* как предварительный шаг перед более точными методами;
* когда время важнее точности.

---

## Схема работы RLSM (для слайда)

### Идея одной картинкой

```
Симуляция траекторий S_t
        │
        ▼
Случайное отображение (Random Features)
φ(S_t) = σ(A · S_t + b)
        │
        ▼
Одна линейная модель
V̂(t, S) = wᵀ φ(S_t)
        │
        ▼
Единая стратегия остановки
(для всех t и всех траекторий)
```

---

### Сравнение со схемой LSM / NLSM

#### LSM / NLSM (пошагово)

```
t = T-1 ──► регрессия
t = T-2 ──► регрессия
t = T-3 ──► регрессия
   ...
t = 0   ──► регрессия
```

❌ много моделей
❌ дорого при большом числе шагов

---

#### RLSM (глобально)

```
Все (t, S_t) ──► одна модель ──► стратегия
```

✅ одна модель
✅ быстро
❌ возможное смещение

---

## Почему это важно подчеркнуть на защите

Очень сильная формулировка (можно почти дословно):

> RLSM вводится не как более точный метод, а как вычислительно эффективная альтернатива LSM и NLSM в задачах с высокой размерностью и большим числом временных шагов.

---

## Короткий ответ комиссии «почему тогда он хуже здесь»

> В нашем эксперименте задача одномерная и чувствительная к точности границы остановки. В таком режиме адаптивные методы, особенно NLSM, имеют преимущество, тогда как сильные стороны RLSM не реализуются.

---

## Если хотите усилить слайд

Можно добавить:

* подпись: **“RLSM ≈ нейросеть с замороженными весами”**
* стрелку: **Speed ↑, Bias ↑**

Если хотите, я могу:

* превратить эту схему в **готовый текст для PowerPoint**;
* помочь сформулировать **один финальный слайд «Когда какой метод использовать»**;
* адаптировать объяснение под **не ML-аудиторию**.


## 18_01_pricing_comparison
Насколько стабильно каждый алгоритм оценивает цену опциона при одинаковых входных данных?
Посмотрим на **распределение оценок цены американского опциона**, полученных:

* по разным прогонам (runs);
* при фиксированных параметрах модели;
* для конкретного алгоритма оптимальной остановки.


## 1. Boxplot: цена опциона vs алгоритм (LSM / RLSM / NLSM)

### Что показывает

Этот boxplot отвечает на вопрос:

> *Насколько стабильно каждый алгоритм оценивает цену опциона при одинаковых входных данных?*

По оси:

* X — алгоритм;
* Y — цена опциона.

---

### Интерпретация элементов

#### Медиана (центральная линия)

* показывает «типичную» оценку цены;
* в Ваших данных медианы всех трёх методов близки.

**Вывод:**

> Все методы корректно решают задачу pricing и не дают систематического смещения.


#### Межквартильный размах (коробка)

* отражает чувствительность алгоритма к случайности траекторий;
* у LSM коробка заметно шире.

**Вывод:**

> LSM обладает высокой вариативностью оценки — цена сильно зависит от конкретной выборки траекторий.


#### Усы и выбросы

* характеризуют экстремальные ошибки;
* у LSM они выражены сильнее.

**Вывод:**

> В отдельных прогонах LSM может давать существенно отличающиеся оценки, что нежелательно для практического применения.

### Итог по этому графику

> Boxplot показывает, что при одинаковой средней цене алгоритмы принципиально различаются по устойчивости, и NLSM даёт наиболее стабильную оценку.


## 2. Boxplot: цена опциона vs число временных шагов ( N )

### Что показывает

Этот график отвечает на вопрос:

> *Как дискретизация времени влияет на стабильность оценки цены?*

По оси:

* X — число временных шагов (10, 25, 50, 100);
* Y — цена опциона.

---

### Интерпретация

#### Рост ширины коробок при увеличении ( N ) (LSM)

* при большем числе шагов:

  * растёт размер регрессионной задачи;
  * ухудшается аппроксимация continuation value.

**Вывод:**

> Классический LSM плохо масштабируется по числу временных шагов.

---

#### Поведение RLSM

* регуляризация частично компенсирует рост размерности;
* разброс растёт медленнее.

**Вывод:**

> Регуляризация повышает устойчивость, но не устраняет проблему полностью.

---

#### Поведение NLSM

* ширина коробок остаётся почти неизменной;
* выбросы минимальны.

**Вывод:**

> Нейросетевая аппроксимация continuation value менее чувствительна к дискретизации времени.

---

### Итог по этому графику

> Boxplot демонстрирует ключевое преимущество нейросетевых методов при мелкой временной сетке.

---

## 3. Boxplot: цена опциона vs число траекторий

### Что показывает

Этот график отвечает на вопрос:

> *Сколько траекторий необходимо каждому методу для стабилизации оценки?*

---

### Интерпретация

#### LSM

* при малом числе траекторий — большой разброс;
* устойчивость достигается только при очень больших выборках.

**Вывод:**

> LSM чувствителен к размеру выборки и требует большого числа траекторий.

---

#### NLSM

* стабилизация происходит быстрее;
* меньшая зависимость от размера выборки.

**Вывод:**

> Нейросетевой метод эффективнее использует данные.

---

### Итог по этому графику

> Boxplot показывает, что NLSM достигает сопоставимой точности при меньшем числе траекторий.

---

## 4. Boxplot по разным прогонам (runs)

### Что показывает

Этот boxplot отвечает на вопрос:

> *Насколько метод чувствителен к инициализации и стохастичности обучения?*

---

### Интерпретация

* LSM:

  * разброс между прогонами значителен;
* NLSM:

  * разброс между прогонами мал.

**Вывод:**

> Нейросетевой метод демонстрирует более воспроизводимые результаты.

---

## Общий мета-вывод по всем boxplot’ам

Вы можете сформулировать так:

> «Все boxplot’ы показывают, что основное различие между алгоритмами заключается не в средней цене опциона, а в устойчивости оценки. Нейросетевой метод демонстрирует меньшую дисперсию, меньшую чувствительность к дискретизации и лучшую воспроизводимость.»

---

## Почему Вы корректно заменяете boxplot’ы таблицей

Финальная логика:

* boxplot’ы → качественный анализ распределений;
* таблица → сжатое представление их статистических свойств;
* это **не потеря информации**, а её структурирование.

---

Если хотите, следующим шагом я могу:

* помочь переписать это объяснение **в 5–6 коротких фраз для устного доклада**;
* связать каждый пункт с **конкретной строкой таблицы**;
* или адаптировать текст под **раздел «Численные эксперименты» в дипломе**.


Ниже — подробное, но «комиссионно-пригодное» объяснение **VaR, CVaR, shortfall и delta**, как они считаются в вашей постановке **one-step hedge**, и готовый текст, как это рассказывать на защите.

---

# 1) Delta (дельта): что это и как считается

## Интуиция

**Дельта** — это чувствительность цены опциона к малому изменению цены базового актива.

Если цена актива чуть выросла на (\Delta S), то цена опциона примерно меняется на:
[
\Delta V \approx \Delta \cdot \Delta S
]

## Формально

Если (V(t,S)) — цена опциона в момент (t) при цене актива (S), то
[
\Delta(t,S) = \frac{\partial V(t,S)}{\partial S}
]

## Как вы считаете (\Delta) в ваших методах

### (a) LSM

У LSM нет гладкой аналитической функции (V(S)), поэтому вы берёте **численную производную**:
[
\Delta \approx \frac{V(t,S+h) - V(t,S-h)}{2h}
]
где (h) — маленький шаг.

### (b) RLSM / NLSM

Continuation value аппроксимируется моделью (регрессия/нейросеть), которая является дифференцируемой функцией входа. Тогда:
[
\Delta = \frac{\partial \hat V(t,S)}{\partial S}
]

* для нейросети это берётся автоматически (autograd),
* для гладкой регрессии — аналитически/автоматически.

**Ключевой смысл:** NLSM и (часто) RLSM дают **более стабильную дельту**, потому что производная получается от сглаженной аппроксимации.

---

# 2) Shortfall: что это и как считается

## Интуиция

Shortfall — это **недостаток денег** у вашего хеджирующего портфеля по сравнению с тем, что нужно, чтобы «перекрыть» обязательство (стоимость опциона) через один шаг.

## В вашей постановке one-step hedge

Вы строите портфель на шаг ([0,\Delta t]):

1. В момент 0:

* у вас есть цена опциона (V_0),
* вы берёте (\Delta_0),
* стартовый капитал (X_0 = V_0).

2. Через один шаг:
   [
   X_1 = X_0 + \Delta_0(S_1 - S_0)
   ]
   (в упрощённой версии без начисления процентов; если учитывать ставку, кэш-компонента растёт как (e^{r\Delta t})).

3. В момент (t_1=\Delta t) опцион «стоит» (V_1) (его цена в этот момент на той же траектории).

## Ошибка хеджа и shortfall

Ошибка (hedging error):
[
\varepsilon = V_1 - X_1
]

Shortfall (только плохая часть):
[
L = (V_1 - X_1)_+ = \max(V_1 - X_1,\ 0)
]

* если (X_1 \ge V_1) → shortfall = 0 (денег хватило)
* если (X_1 < V_1) → shortfall = дефицит

**Почему shortfall важнее ошибки:** он фокусируется именно на риске *недохеджа* (когда денег не хватило), а не на случаях «перехеджировали».

---

# 3) VaR: что это и как считается

## Интуиция

**VaR на уровне 0.99** — это число (v), такое что в 99% случаев shortfall **не превышает** (v).

Иначе:

> «В худших 1% сценариев shortfall может быть больше этого числа».

## Формально

[
\mathrm{VaR}_{0.99}(L) = \inf{v:\ \mathbb{P}(L \le v)\ge 0.99}
]

## Как вы считаете на траекториях (эмпирически)

У вас есть (N) траекторий → массив (L^{(1)},\dots,L^{(N)}).

Алгоритм:

1. отсортировать (L),
2. взять квантиль:
   [
   \mathrm{VaR}*{0.99} \approx L*{(\lceil 0.99 N\rceil)}
   ]

**Смысл VaR:** это «порог» tail-риска, но он не говорит, что происходит *за* этим порогом.

---

# 4) CVaR: что это и как считается

## Интуиция

**CVaR (Expected Shortfall)** на уровне 0.99 — это **средний shortfall в худших 1% случаев**.

Это более строгая и информативная метрика, чем VaR, потому что учитывает величину потерь в хвосте.

## Формально

[
\mathrm{CVaR}*{0.99}(L) = \mathbb{E}\left[L\ \middle|\ L \ge \mathrm{VaR}*{0.99}(L)\right]
]

## Эмпирически

1. находите VaR,
2. берёте все траектории, где (L) в топ-1%,
3. усредняете по ним.

---

# 5) Что такое “hedging risk” в вашей работе

В вашем дипломе **hedging risk** — это риск недохеджа на один шаг, измеренный через tail-метрики распределения shortfall:

* (\mathrm{VaR}_{0.99}(L))
* (\mathrm{CVaR}_{0.99}(L))

То есть:

> «Какой дефицит денег может возникнуть в плохих сценариях при одном шаге ребалансировки».

---

# 6) Как рассказать это комиссии (готовый текст)

Ниже — версия, которую обычно хорошо воспринимают.

### Вариант «коротко и уверенно» (40–60 секунд)

> «Для хеджирования нам нужна дельта — производная цены опциона по цене базового актива, (\Delta=\partial V/\partial S). В момент времени 0 я беру оценку цены (V_0) и дельту (\Delta_0) и строю one-step delta hedge: (X_1 = V_0 + \Delta_0(S_1-S_0)). Далее на каждой траектории считаю shortfall — недохедж: (L=(V_1-X_1)_+), то есть дефицит денег относительно стоимости опциона через шаг. По распределению (L) оцениваю tail-риск: VaR_{0.99} — это 99-й квантиль shortfall, а CVaR_{0.99} — средний shortfall в худших 1% сценариев. Это и есть количественный результат хеджа на один шаг: насколько стратегия устойчива в хвостах распределения.»

### Вариант «если начинают уточнять» (ещё 30 секунд)

> «VaR говорит порог потерь, но не описывает размер потерь за порогом, поэтому CVaR более строгий. Именно поэтому я показываю оба. Нейросетевой метод даёт аналитическую дельту как производную модели, а в LSM дельта получается численно, поэтому tail-риски у нейросети обычно ниже.»

---

# 7) Что вынести на слайд (минимум формул)

На слайде достаточно 4 строк:

1. (\Delta = \partial V/\partial S)
2. (X_1 = V_0 + \Delta_0(S_1-S_0))
3. (L=(V_1-X_1)_+)
4. (\mathrm{VaR}*{0.99}(L)), (\mathrm{CVaR}*{0.99}(L))

И 1 подпись:

> «Tail-риск недохеджа на один шаг».

---

Если хотите, я могу адаптировать этот блок под **вашу конкретную терминологию из ноутбука** (например, если у вас в коде shortfall задан относительно buy-and-hold или по иной нормировке) и написать формулировку так, чтобы она **однозначно совпадала с вычислениями**.



$
\Delta = \partial V/\partial S 
$

$
X_1 = V_0 + \Delta_0(S_1-S_0)
$

$
L=(V_1-X_1)_+
$

$
\mathrm{VaR}*{0.99}(L), \mathrm{CVaR}*{0.99}(L)
$